In [1]:
# --- edit these ---
MODEL = "fireworks_ai/accounts/fireworks/models/glm-5p2"            # model under test
# Only glm-5p1 is serverless on this account, so it self-judges tool-call equivalence.
JUDGE_MODEL = "fireworks_ai/accounts/fireworks/models/glm-5p2"
TEMPERATURE = 0.0
MAX_TOKENS = 8192
MAX_ROWS = 30        # tool sets do gen+judge (both GLM reasoning) -> keep small
CONCURRENCY = 6
REQUEST_TIMEOUT = 900
MCP_CONFIG_PATH = ""  # set to a valid MCP config path to enable real tool execution

# Tool / agent calling sets use an LLM judge (semantic match, format-agnostic).
# Math sets (grade locally): gsm8k, math500_l5, aime2024, aime2025, hmmt2025
RUN = ["toolace"]

In [2]:
import asyncio
import inspect
import json
import os
import re
from pathlib import Path

import litellm
from datasets import load_dataset
from dotenv import load_dotenv
from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import SingleTurnRolloutProcessor
from eval_protocol.pytest.types import RolloutProcessorConfig

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")
if not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")
litellm.drop_params = True

MATH_SYSTEM = (
    "You are a careful mathematician. Work step by step, then give the final "
    "answer on the last line as \\boxed{...}."
)
TOOL_SYSTEM = (
    "You are a function-calling assistant.\n"
    "Available tools:\n{tools}"
)

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TaskDefinitionModel(BaseModel):


In [3]:
_BOXED_RE = re.compile(r"\\boxed\s*\{", re.DOTALL)
_NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")


def extract_boxed(text):
    m = list(_BOXED_RE.finditer(text or ""))
    if not m:
        return None
    start, depth, i = m[-1].end(), 1, m[-1].end()
    while i < len(text) and depth > 0:
        depth += {"{": 1, "}": -1}.get(text[i], 0)
        i += 1
    return text[start : i - 1].strip() if depth == 0 else None


def model_response(row):
    return str(row.messages[-1].content) if row.messages else ""


def user_text(row):
    for m in row.messages:
        if getattr(m, "role", None) == "user":
            return str(m.content)
    return ""


def grade_math(row):
    text = model_response(row)
    pred = extract_boxed(text)
    if pred is None:
        nums = _NUM_RE.findall(text.replace(",", ""))
        pred = nums[-1] if nums else None
    gt = str(row.ground_truth)
    if pred is None:
        return EvaluateResult(score=0.0, reason="no answer parsed")
    try:
        if abs(float(pred.replace(",", "")) - float(gt.replace(",", ""))) < 1e-6:
            return EvaluateResult(score=1.0, reason=f"num pred={pred} gt={gt}")
    except (ValueError, OverflowError):
        pass
    try:
        from math_verify import parse, verify
        g, p = parse(f"\\boxed{{{gt}}}"), parse(f"\\boxed{{{pred}}}")
        if g and p and verify(g, p):
            return EvaluateResult(score=1.0, reason=f"mv pred={pred} gt={gt}")
    except Exception:
        pass
    return EvaluateResult(score=0.0, reason=f"pred={pred} gt={gt}")

In [4]:
JUDGE_SEM = asyncio.Semaphore(CONCURRENCY)

_JUDGE_PROMPT = (
    "You grade a function-calling answer. Reply with ONLY one word: CORRECT or INCORRECT.\n\n"
    "User request:\n{query}\n\n"
    "Reference correct call(s):\n{ref}\n\n"
    "Model's answer:\n{ans}\n\n"
    "Mark CORRECT iff the model calls the same function(s) as the reference with semantically "
    "equivalent arguments. Ignore formatting, key order, quoting, and date/number formatting. "
    "If the reference indicates NO function should be called (empty list / none), the model is "
    "CORRECT only if it also makes no function call."
)


async def grade_tool_judge(row):
    ans = model_response(row)[-2500:]
    prompt = _JUDGE_PROMPT.format(query=user_text(row)[:1500], ref=str(row.ground_truth)[:1500], ans=ans)
    try:
        async with JUDGE_SEM:
            resp = await litellm.acompletion(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0, max_tokens=1024, timeout=180,
            )
        verdict = (resp.choices[0].message.content or "").strip().upper()
    except Exception as e:
        return EvaluateResult(score=0.0, reason=f"judge error: {str(e)[:40]}")
    ok = "CORRECT" in verdict and "INCORRECT" not in verdict
    return EvaluateResult(score=1.0 if ok else 0.0, reason=verdict[:30])

In [5]:
def _seg(conv, names):
    for t in conv:
        role = t.get("role", t.get("from"))
        if role in names:
            return t.get("content", t.get("value"))
    return None


def _coerce_tools(tools):
    if not tools:
        return None
    if isinstance(tools, str):
        try:
            tools = json.loads(tools)
        except Exception:
            return None

    if isinstance(tools, dict):
        if isinstance(tools.get("tools"), list):
            tools = tools["tools"]
        else:
            return None

    if not isinstance(tools, list):
        return None

    normalized = []
    for t in tools:
        if not isinstance(t, dict):
            continue
        if t.get("type") == "function" and isinstance(t.get("function"), dict) and t["function"].get("name"):
            normalized.append(t)
            continue

        name = t.get("name")
        if not name:
            continue
        fn = {"name": name}
        if t.get("description"):
            fn["description"] = t["description"]
        params = t.get("parameters") or t.get("input_schema") or t.get("args_schema")
        if params:
            fn["parameters"] = params
        normalized.append({"type": "function", "function": fn})

    return normalized or None


def _math_row(q, gt):
    return EvaluationRow(messages=[{"role": "system", "content": MATH_SYSTEM},
                                   {"role": "user", "content": q}], ground_truth=gt)


def _tool_row(system, query, ref, tools=None):
    row = EvaluationRow(messages=[{"role": "system", "content": system},
                                  {"role": "user", "content": query}], ground_truth=ref)
    row.tools = _coerce_tools(tools)
    return row


# ---- math ----
def build_gsm8k(n):
    ds = load_dataset("openai/gsm8k", "main", split="test")
    return [_math_row(r["question"], r["answer"].split("####")[-1].strip().replace(",", "")) for r in list(ds)[:n]]
def build_math500_l5(n):
    ds = [r for r in load_dataset("HuggingFaceH4/MATH-500", split="test") if r["level"] == 5]
    return [_math_row(r["problem"], str(r["answer"])) for r in ds[:n]]
def build_aime2024(n):
    ds = load_dataset("Maxwell-Jia/AIME_2024", split="train")
    return [_math_row(r["Problem"], str(r["Answer"])) for r in list(ds)[:n]]
def build_aime2025(n):
    ds = load_dataset("MathArena/aime_2025", split="train")
    return [_math_row(r["problem"], str(r["answer"])) for r in list(ds)[:n]]
def build_hmmt2025(n):
    ds = load_dataset("MathArena/hmmt_feb_2025", split="train")
    return [_math_row(r["problem"], str(r["answer"])) for r in list(ds)[:n]]


# ---- tool / agent (judge-graded) ----
def build_xlam60k(n):
    ds = load_dataset("minpeter/xlam-function-calling-60k-parsed", "xlam-function-calling-60k", split="train")
    rows = []
    for r in list(ds)[:n]:
        msgs = r["messages"]
        query = next((m["content"] for m in msgs if m["role"] == "user"), None)
        asst = next((m for m in msgs if m["role"] == "assistant"), None)
        if not query or not asst:
            continue
        ref = json.dumps([{"name": tc["function"]["name"], "arguments": tc["function"]["arguments"]}
                          for tc in (asst.get("tool_calls") or [])])
        rows.append(_tool_row(TOOL_SYSTEM.format(tools=str(r.get("tools", ""))[:2000]), query, ref, tools=r.get("tools")))
    return rows

def build_hermes(n):
    ds = load_dataset("NousResearch/hermes-function-calling-v1", "func_calling_singleturn", split="train")
    rows = []
    for r in list(ds)[:n]:
        conv = r["conversations"]
        system, query, ref = _seg(conv, ["system"]), _seg(conv, ["human", "user"]), _seg(conv, ["gpt", "assistant"])
        if system and query and ref:
            rows.append(_tool_row(system, query, ref, tools=r.get("tools")))
    return rows

def build_toolace(n):
    ds = load_dataset("Team-ACE/ToolACE", split="train")
    rows = []
    for r in list(ds)[:n]:
        query, ref = _seg(r["conversations"], ["user"]), _seg(r["conversations"], ["assistant"])
        tools = r.get("tools")
        system = r["system"]
        if not tools and "{tools}" in system:
            system = system.format(tools="")
        if query and ref:
            rows.append(_tool_row(system, query, ref, tools=tools))
    return rows

def build_xlam_irrelevance(n):
    ds = load_dataset("MadeAgents/xlam-irrelevance-7.5k", split="train")
    rows = []
    for r in list(ds)[:n]:
        rows.append(_tool_row(TOOL_SYSTEM.format(tools=str(r.get("tools", ""))[:2000]), r["query"], r["answers"], tools=r.get("tools")))
    return rows

def build_driaforall(n):
    ds = load_dataset("driaforall/pythonic-function-calling", split="train")
    rows = []
    for r in list(ds)[:n]:
        conv = r["conversations"]
        system = _seg(conv, ["system"]) or "You write Python calling the provided functions."
        query, ref = _seg(conv, ["user", "human"]), _seg(conv, ["assistant", "gpt"])
        if query and ref:
            rows.append(_tool_row(system + "\n\nFunctions:\n" + str(r.get("tools", ""))[:2000], query, ref, tools=r.get("tools")))
    return rows

def build_nexusraven(n):
    apis = load_dataset("Nexusflow/NexusRaven_API_evaluation", "standardized_api_list", split="train")
    by_name = {a["name"]: {"desc": (a["description"] or "").strip()[:200],
                           "args": [d.get("name") for d in a["args_dicts"] if d.get("name")]} for a in apis}
    queries = load_dataset("Nexusflow/NexusRaven_API_evaluation", "standardized_queries", split="train")
    rows = []
    for q in list(queries)[:n]:
        tool_defs = []
        sigs = []
        for fn in (q["context_functions"] or []):
            if fn not in by_name:
                continue
            args = by_name[fn]["args"]
            sigs.append(f"- {fn}({', '.join(args)}): {by_name[fn]['desc']}")
            tool_defs.append({
                "type": "function",
                "function": {
                    "name": fn,
                    "description": by_name[fn]["desc"],
                    "parameters": {
                        "type": "object",
                        "properties": {a: {"type": "string"} for a in args},
                    },
                },
            })
        ref = json.dumps({"name": q["python_function_name"], "arguments": json.loads(q["python_args_dict"])})
        rows.append(_tool_row(TOOL_SYSTEM.format(tools="\n".join(sigs)), q["prompt"], ref, tools=tool_defs))
    return rows


BENCHMARKS = {
    "gsm8k": (build_gsm8k, grade_math),
    "math500_l5": (build_math500_l5, grade_math),
    "aime2024": (build_aime2024, grade_math),
    "aime2025": (build_aime2025, grade_math),
    "hmmt2025": (build_hmmt2025, grade_math),
    "xlam60k": (build_xlam60k, grade_tool_judge),            # xLAM function calling (parallel/multi)
    "hermes": (build_hermes, grade_tool_judge),              # Hermes single-turn function calling
    "toolace": (build_toolace, grade_tool_judge),            # ToolACE (compositional, hard)
    "xlam_irrelevance": (build_xlam_irrelevance, grade_tool_judge),  # relevance detection (should NOT call)
    "driaforall": (build_driaforall, grade_tool_judge),      # pythonic function calling
    "nexusraven": (build_nexusraven, grade_tool_judge),      # API calling (CVE/email/...)
}

In [6]:
async def run_benchmark(rows, grader):
    processor = SingleTurnRolloutProcessor(drop_trailing_assistant_messages=True)
    config = RolloutProcessorConfig(
        completion_params={"model": MODEL, "temperature": TEMPERATURE,
                           "max_tokens": MAX_TOKENS, "timeout": REQUEST_TIMEOUT},
        mcp_config_path=MCP_CONFIG_PATH,
        semaphore=asyncio.Semaphore(CONCURRENCY),
    )
    results = await asyncio.gather(*processor(rows, config), return_exceptions=True)
    ev_rows = [r for r in results if isinstance(r, EvaluationRow)]
    errors = len(results) - len(ev_rows)

    async def _grade(r):
        out = grader(r)
        if inspect.isawaitable(out):
            out = await out
        r.evaluation_result = out
        return r

    graded = await asyncio.gather(*[_grade(r) for r in ev_rows])
    return graded, errors

In [7]:
summary = {}
for name in RUN:
    build, grader = BENCHMARKS[name]
    rows = build(MAX_ROWS)
    for i, r in enumerate(rows):
        r.input_metadata.row_id = f"{name}-{i}"
    print(f"{name}: running {len(rows)} rows ({grader.__name__})...")
    graded, errors = await run_benchmark(rows, grader)
    scores = [g.evaluation_result.score for g in graded if g.evaluation_result]
    acc = sum(scores) / len(scores) if scores else 0.0
    summary[name] = (acc, int(sum(scores)), len(scores), errors)
    print(f"  {name}: {acc:.1%} ({int(sum(scores))}/{len(scores)})"
          + (f"  [{errors} dropped]" if errors else ""))
    for g in graded[:3]:
        print(f"     [{g.evaluation_result.score:.0f}] {g.evaluation_result.reason[:70]}")

print("\n=== GLM-5.1 base accuracy ===")
for name, (acc, c, t, e) in summary.items():
    flag = "  <- RL target (10-50%)" if 0.10 <= acc <= 0.50 else ""
    print(f"  {name:18s}: {acc:6.1%}  ({c}/{t}){'  ['+str(e)+' dropped]' if e else ''}{flag}")

toolace: running 29 rows (grade_tool_judge)...
  toolace: 79.3% (23/29)
     [0] INCORRECT
     [1] CORRECT
     [1] CORRECT

=== GLM-5.1 base accuracy ===
  toolace           :  79.3%  (23/29)


In [8]:
def _msg_role(m):
    return getattr(m, "role", None) if not isinstance(m, dict) else m.get("role")

def _msg_content(m):
    return getattr(m, "content", None) if not isinstance(m, dict) else m.get("content")

def _msg_tool_calls(m):
    if isinstance(m, dict):
        return m.get("tool_calls")
    return getattr(m, "tool_calls", None)

def _msg_tool_call_id(m):
    if isinstance(m, dict):
        return m.get("tool_call_id")
    return getattr(m, "tool_call_id", None)

def inspect_tool_usage(graded_rows, k=5):
    with_calls = 0
    with_tool_msgs = 0

    for i, r in enumerate(graded_rows):
        msgs = list(r.messages or [])
        has_calls = any(_msg_tool_calls(m) for m in msgs if _msg_role(m) == "assistant")
        has_tool_msgs = any(_msg_role(m) == "tool" for m in msgs)
        with_calls += int(has_calls)
        with_tool_msgs += int(has_tool_msgs)

        if i < k:
            print(f"\n--- row {i} ---")
            print(f"score={getattr(r.evaluation_result, 'score', None)} reason={getattr(r.evaluation_result, 'reason', '')[:80]}")
            print(f"input tools attached? {bool(getattr(r, 'tools', None))}")
            print(f"assistant tool_calls? {has_calls} | tool messages? {has_tool_msgs}")
            for j, m in enumerate(msgs[-4:]):  # last turns
                role = _msg_role(m)
                content = str(_msg_content(m) or "")[:200]
                tc = _msg_tool_calls(m)
                tcid = _msg_tool_call_id(m)
                print(f"  [{j}] role={role} tool_call_id={tcid} tool_calls={bool(tc)} content={content!r}")
                if tc:
                    print(f"      tool_calls={tc}")

    n = len(graded_rows)
    print("\n=== tool usage summary ===")
    print(f"assistant emitted tool_calls: {with_calls}/{n} ({(with_calls/n if n else 0):.1%})")
    print(f"tool role messages present:   {with_tool_msgs}/{n} ({(with_tool_msgs/n if n else 0):.1%})")

inspect_tool_usage(graded, k=8)


--- row 0 ---
score=0.0 reason=INCORRECT
input tools attached? False
assistant tool_calls? False | tool messages? False
  [0] role=system tool_call_id=None tool_calls=False content='You are an expert in composing functions. You are given a question and a set of possible functions. \nBased on the question, you will need to make one or more function/tool calls to achieve the purpose'
  [1] role=user tool_call_id=None tool_calls=False content="I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?"
  [2] role=assistant tool_call_id=None tool_calls=False content="[Market Trends API(trend_type='MARKET_INDEXES', country='us', language='en'), Market Trends API(trend_type='MOST_ACTIVE', country='us', language='en')]"

--- row 1 ---
score=1.0 reason=CORRECT
input tools attached? False
assistant tool_calls? False | tool messages? False
  [0] role=system tool_call_id=None tool_calls=False content='You are an expe